In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# Phase 2 — HE-Friendly UNet on ACDC\n", "Training with PolyAct + BatchNorm + pretrained init on Colab GPU."]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Mount Drive\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Clone repo\n",
    "!git clone https://github.com/masiirene/acdc-he-segmentation.git\n",
    "%cd acdc-he-segmentation\n",
    "!pip install nibabel -q"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Copy data to RAM for speed\n",
    "!mkdir -p /content/acdc_data\n",
    "!cp -r /content/drive/MyDrive/tesi/tesi_acdc/training /content/acdc_data/\n",
    "!cp /content/drive/MyDrive/tesi/splits_final.json /content/acdc_data/\n",
    "!mkdir -p data\n",
    "!cp /content/drive/MyDrive/tesi/baseline_weights.pth data/\n",
    "print('Data ready!')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import subprocess, sys\n",
    "sys.path.insert(0, '/content/acdc-he-segmentation')\n",
    "\n",
    "# Verify GPU\n",
    "import torch\n",
    "print(f'GPU: {torch.cuda.get_device_name(0)}')\n",
    "print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Launch training with batch_size=32\n",
    "!python training/train.py \\\n",
    "    --act poly \\\n",
    "    --norm batch \\\n",
    "    --pretrained data/baseline_weights.pth \\\n",
    "    --batch_size 32 \\\n",
    "    --lr 1e-4 \\\n",
    "    --epochs 150 \\\n",
    "    --early_stop 100 \\\n",
    "    --data_dir /content/acdc_data/training \\\n",
    "    --out_dir results/colab_bs32"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save results to Drive\n",
    "!mkdir -p /content/drive/MyDrive/tesi/phase2_results\n",
    "!cp -r results/colab_bs32 /content/drive/MyDrive/tesi/phase2_results/\n",
    "print('Results saved to Drive!')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluate best model\n",
    "import os, torch, numpy as np\n",
    "from torch.utils.data import DataLoader\n",
    "sys.path.insert(0, '/content/acdc-he-segmentation')\n",
    "from models.he_friendly import HEFriendlyUNet\n",
    "from training.dataset import ACDCDataset, load_splits\n",
    "\n",
    "device = torch.device('cuda')\n",
    "model = HEFriendlyUNet(act_type='poly', norm_type='batch').to(device)\n",
    "\n",
    "best_path = 'results/colab_bs32/act=poly_norm=batch_bs32_lr0.0001/best_model.pth'\n",
    "model.load_state_dict(torch.load(best_path, map_location='cpu', weights_only=False))\n",
    "model.eval()\n",
    "\n",
    "train_cases, val_cases = load_splits('/content/acdc_data/splits_final.json', fold=0)\n",
    "ds = ACDCDataset('/content/acdc_data/training', val_cases, patch_size=(256,224), augment=False)\n",
    "loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2)\n",
    "\n",
    "dice_rv, dice_myo, dice_lv = [], [], []\n",
    "with torch.no_grad():\n",
    "    for imgs, segs in loader:\n",
    "        imgs, segs = imgs.to(device), segs.to(device)\n",
    "        preds = model(imgs).argmax(dim=1)\n",
    "        for label, lst in [(1,dice_rv),(2,dice_myo),(3,dice_lv)]:\n",
    "            p2 = (preds==label).float()\n",
    "            t = (segs==label).float()\n",
    "            score = (2*(p2*t).sum()+1e-5)/(p2.sum()+t.sum()+1e-5)\n",
    "            lst.append(score.item())\n",
    "\n",
    "print(f'RV:  {sum(dice_rv)/len(dice_rv):.3f}')\n",
    "print(f'MYO: {sum(dice_myo)/len(dice_myo):.3f}')\n",
    "print(f'LV:  {sum(dice_lv)/len(dice_lv):.3f}')\n",
    "print(f'Mean: {(sum(dice_rv)+sum(dice_myo)+sum(dice_lv))/(3*len(dice_rv)):.3f}')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}